# 🍲 FoodShareAI — YOLOv8n Food Detection Cloud GPU Fine-Tuning
### Complete End-to-End Workflow for Google Colab (Free Tier GPU)

> **CRITICAL PROJECT CONSTRAINTS & AUTHORIZATION BOUNDARY**:
> - **Budget**: Strictly **FREE ONLY** (Google Colab T4 Free Tier, zero credit card, zero subscriptions).
> - **Production Safety**: Never overwrite or modify `ml-service/models/food_detection/best.pt`.
> - **Model Promotion**: Candidate checkpoints are evaluated for recommendation only; manual user approval is strictly required before any production deployment.
> - **Test Split Policy**: The 300-image test split is strictly held-out. Never use the test split for training or hyperparameter tuning.


## 📋 Step-by-Step Execution Guide for Users

1. **Upload Files to Google Drive**:
   - Create a folder named `FoodShareAI` in your Google Drive root.
   - Upload `food_detection_dataset.zip` (~797 MB) into `Google Drive > FoodShareAI >`.
   - Upload `best_epoch5_checkpoint.pt` (~23.4 MB) into `Google Drive > FoodShareAI >`.
2. **Open Notebook in Google Colab**:
   - In Colab, click `Runtime > Change runtime type > Hardware accelerator > T4 GPU`.
3. **Execute Preflight Cells (Sections 1 through 4)**:
   - Mount Google Drive, verify GPU allocation, extract & validate dataset, and verify checkpoint compatibility.
4. **Approve & Execute Training (Section 5)**:
   - Change `START_TRAINING = True` in Section 5 only when ready.
   - Training runs 20 additional epochs on GPU (~10–15 minutes total).
   - Checkpoints and logs are continuously saved directly to Google Drive.
5. **Run Validation & Evaluation (Sections 6 through 8)**:
   - Evaluate against the 589 validation images and compare against the Epoch 5 baseline.


---
## 1. Setup & GPU Preflight Check
This cell verifies that a free NVIDIA GPU (typically a T4) is allocated and prints library versions.
If no GPU is detected, execution **halts cleanly** without incurring any cost.


In [ ]:
# Cell 1: Environment & Free GPU Preflight
import sys
import os
import shutil
import platform
import subprocess

print("=" * 65)
print("🚀 FoodShareAI Cloud Preflight Environment Check")
print("=" * 65)

# Check Python version
print(f"Python Version: {platform.python_version()}")

# Install pinned Ultralytics without upgrading PyTorch
try:
    import ultralytics
    print(f"Ultralytics Version (Pre-installed): {ultralytics.__version__}")
except ImportError:
    print("Installing Ultralytics (pinned compatible release)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.40"])
    import ultralytics
    print(f"Ultralytics Version (Installed): {ultralytics.__version__}")

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print("\n" + "!" * 65)
    print("❌ NO NVIDIA GPU DETECTED!")
    print("Please navigate to: 'Runtime' -> 'Change runtime type' -> Select 'T4 GPU' (Free Tier).")
    print("Execution halted cleanly to protect free budget. Do NOT continue on CPU.")
    print("!" * 65)
    raise SystemExit("Halting: GPU required for training. Reconnect with GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"\n✅ GPU Allocated:  {gpu_name}")
print(f"✅ VRAM Capacity: {gpu_mem_gb:.2f} GB")

# Hardware-aware training configuration based on detected free GPU
if gpu_mem_gb >= 14.0:
    # Standard Colab T4 (15.0 - 16.0 GB VRAM)
    RECOMMENDED_BATCH_SIZE = 16
    RECOMMENDED_IMGSZ = 640
elif gpu_mem_gb >= 8.0:
    RECOMMENDED_BATCH_SIZE = 8
    RECOMMENDED_IMGSZ = 640
elif gpu_mem_gb >= 4.0:
    RECOMMENDED_BATCH_SIZE = 4
    RECOMMENDED_IMGSZ = 640
else:
    print("\n" + "!" * 65)
    print("❌ GPU VRAM INSUFFICIENT (< 4 GB) for 640px YOLOv8n fine-tuning!")
    print("Execution halted cleanly to protect free budget.")
    print("!" * 65)
    raise SystemExit("Halting: Insufficient GPU VRAM for 640px training.")

print(f"🎯 Hardware Batch Size: {RECOMMENDED_BATCH_SIZE} (Conservative for detected VRAM)")
print(f"🎯 Image Resolution:    {RECOMMENDED_IMGSZ} px")

# Check disk space
total, used, free = shutil.disk_usage("/")
free_gb = free / (1024 ** 3)
print(f"✅ Colab Disk Free:      {free_gb:.2f} GB available")
if free_gb < 10.0:
    print("⚠️ Low disk space warning: less than 10 GB available.")
print("=" * 65)


---
## 2. Google Drive Mounting & Experiment Persistence
Mount your Google Drive to ensure all checkpoints and logs are automatically preserved across potential disconnects.


In [ ]:
# Cell 2: Google Drive Mounting & Directory Setup
from google.colab import drive
from pathlib import Path
import os
import sys
import shutil

print("Mounting Google Drive (Authorize access when prompted by Google)...")
drive.mount('/content/drive')

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FoodShareAI')
DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_EXPERIMENTS_DIR = DRIVE_PROJECT_ROOT / 'experiments'
DRIVE_EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✅ Google Drive Mounted Successfully!")
print(f"📁 Project Storage Root: {DRIVE_PROJECT_ROOT}")
print(f"📁 Experiments Output:   {DRIVE_EXPERIMENTS_DIR}")

# Intelligent multi-location file locator
# Checks FoodShareAI folder first, falls back to MyDrive root or /content/, and auto-organizes
required_files = {
    'food_detection_dataset.zip': {
        'desc': 'Dataset Archive (2,950 images + labels)',
        'expected_sha256': 'C3B01ACE73F7598B0C594FC7F4BEAD7DE5BAA6D667A632A5E3D57C9E4917E94C'
    },
    'best_epoch5_checkpoint.pt': {
        'desc': 'Epoch 5 Baseline Checkpoint (YOLOv8n)',
        'expected_sha256': '1DA8CD2C9CC6911108DA61456B5E5FC7D6BD84D1C79A24DC18E2321A602FAF6B'
    }
}

candidate_search_dirs = [
    DRIVE_PROJECT_ROOT,
    Path('/content/drive/MyDrive'),
    Path('/content')
]

print("\n" + "=" * 65)
print("🔍 Searching for Required Files in Google Drive & Colab:")
print("=" * 65)

missing_files = []
for fname, info in required_files.items():
    target_path = DRIVE_PROJECT_ROOT / fname
    if target_path.exists():
        size_mb = round(target_path.stat().st_size / (1024 * 1024), 2)
        print(f" ✅ {fname:28s} READY in FoodShareAI folder ({size_mb} MB)")
        continue
    
    # Check fallback search directories
    found_in_fallback = None
    for sdir in candidate_search_dirs[1:]:
        fallback_path = sdir / fname
        if fallback_path.exists():
            found_in_fallback = fallback_path
            break
            
    if found_in_fallback:
        print(f" ℹ️  Found {fname} in {found_in_fallback.parent}!")
        print(f"    Moving to {DRIVE_PROJECT_ROOT} for proper persistence...")
        try:
            shutil.move(str(found_in_fallback), str(target_path))
            size_mb = round(target_path.stat().st_size / (1024 * 1024), 2)
            print(f" ✅ {fname:28s} Relocated successfully ({size_mb} MB)")
        except Exception as e:
            # If cross-device move fails, copy instead
            shutil.copy2(str(found_in_fallback), str(target_path))
            size_mb = round(target_path.stat().st_size / (1024 * 1024), 2)
            print(f" ✅ {fname:28s} Copied to FoodShareAI folder ({size_mb} MB)")
    else:
        print(f" ❌ {fname:28s} NOT FOUND in FoodShareAI, MyDrive root, or /content/")
        missing_files.append((fname, target_path, info['desc']))

print("=" * 65)

if missing_files:
    print("\n" + "!" * 65)
    print("⛔ CRITICAL: MISSING REQUIRED ASSETS!")
    print("Please upload the following file(s) into your Google Drive 'FoodShareAI' folder:\n")
    for fname, fpath, desc in missing_files:
        print(f"  • File: {fname}")
        print(f"    Target:      {fpath}")
        print(f"    Description: {desc}\n")
    print("After uploading, re-run this cell to continue.")
    print("!" * 65)
    raise FileNotFoundError(f"Missing required asset(s): {[m[0] for m in missing_files]}")
else:
    print("🎉 All required assets verified in Google Drive! Ready for extraction.")


---
## 3. Dataset Extraction & Validation
Extracts the dataset onto the fast local NVMe runtime disk (`/content/dataset/`) to eliminate Google Drive I/O bottlenecks during training.


In [ ]:
# Cell 3: Fast Local NVMe Extraction & Integrity Verification
import zipfile
import hashlib
import json
import yaml
from pathlib import Path

LOCAL_DATASET_ROOT = Path('/content/dataset')
LOCAL_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

ZIP_SOURCE = DRIVE_PROJECT_ROOT / 'food_detection_dataset.zip'
if not ZIP_SOURCE.exists():
    raise FileNotFoundError(f"Dataset ZIP not found at {ZIP_SOURCE}. Please upload it to your FoodShareAI folder.")

print(f"Extracting {ZIP_SOURCE.name} to local runtime disk ({LOCAL_DATASET_ROOT})...")
with zipfile.ZipFile(ZIP_SOURCE, 'r') as zf:
    zf.extractall(LOCAL_DATASET_ROOT)

DATASET_DIR = LOCAL_DATASET_ROOT / 'food_detection'
print(f"✅ Extraction complete at {DATASET_DIR}")

# Verify file counts
splits = ['train', 'val', 'test']
expected_counts = {'train': 2061, 'val': 589, 'test': 300}
found_counts = {}

print("\nVerifying dataset splits & 1-to-1 image-label pairings:")
for split in splits:
    img_files = sorted(list((DATASET_DIR / 'images' / split).glob('*.*')))
    lbl_files = sorted(list((DATASET_DIR / 'labels' / split).glob('*.txt')))
    img_count = len(img_files)
    lbl_count = len(lbl_files)
    found_counts[split] = {'images': img_count, 'labels': lbl_count}
    print(f" - {split.upper():5s}: {img_count} images, {lbl_count} labels (Expected: {expected_counts[split]})")
    assert img_count == expected_counts[split], f"Mismatch in {split} images: {img_count} != {expected_counts[split]}"
    assert lbl_count == expected_counts[split], f"Mismatch in {split} labels: {lbl_count} != {expected_counts[split]}"

# Generate Linux-compatible dataset YAML pointing directly to local extraction
colab_yaml_path = DATASET_DIR / 'food_detection_colab.yaml'
dataset_cfg = {
    'path': str(DATASET_DIR.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'Pizza', 1: 'Hamburger', 2: 'Sandwich', 3: 'Hot dog', 4: 'Pasta',
        5: 'Salad', 6: 'Sushi', 7: 'Taco', 8: 'Apple', 9: 'Banana',
        10: 'Orange (fruit)', 11: 'Strawberry', 12: 'Grape', 13: 'Tomato', 14: 'Carrot',
        15: 'Broccoli', 16: 'Bread', 17: 'Cake', 18: 'Cookie', 19: 'Croissant',
        20: 'Cheese', 21: 'Milk', 22: 'Ice cream', 23: 'Coffee (drink)', 24: 'Tea',
        25: 'Juice', 26: 'Wine', 27: 'Popcorn', 28: 'Candy', 29: 'Egg'
    }
}

with open(colab_yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(dataset_cfg, f, sort_keys=False)

print(f"\n✅ Linux Dataset YAML created at: {colab_yaml_path}")
print("✅ Verified 2,061 train images, 589 val images, 300 held-out test images across all 30 classes.")


---
## 4. Checkpoint Verification & Compatibility
Verifies the experimental Epoch 5 checkpoint before training.

> **Fine-Tuning vs. Resuming**:
> - **Fine-Tuning (Model Re-Initialization)**: Loads the weights from `best.pt` into YOLOv8n and starts a new training session with fresh learning rate scheduling and augmentations.
> - **Resuming (`resume=True`)**: Attempts to continue with the exact frozen optimizer states from `last.pt`.
>
> For cloud GPU fine-tuning, **fine-tuning from `best.pt`** with updated batch size (16) and resolution (640) is recommended over continuing with CPU optimizer states.


In [ ]:
# Cell 4: Checkpoint Integrity & Class Ordering Verification
import torch
import hashlib
from ultralytics import YOLO
from pathlib import Path

CKPT_SOURCE = DRIVE_PROJECT_ROOT / 'best_epoch5_checkpoint.pt'
if not CKPT_SOURCE.exists():
    raise FileNotFoundError(f"Checkpoint not found at {CKPT_SOURCE}. Please upload it to your FoodShareAI folder.")

def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192 * 1024):
            h.update(chunk)
    return h.hexdigest().upper()

print(f"Loading {CKPT_SOURCE.name}...")
model = YOLO(str(CKPT_SOURCE))

ckpt_hash = compute_sha256(CKPT_SOURCE)
EXPECTED_CKPT_HASH = "1DA8CD2C9CC6911108DA61456B5E5FC7D6BD84D1C79A24DC18E2321A602FAF6B"

print("\nModel Architecture & Integrity Information:")
print(f" - Model Type:      {type(model.model).__name__}")
print(f" - Class Count:     {len(model.names)}")
print(f" - Checkpoint Size: {round(CKPT_SOURCE.stat().st_size / (1024*1024), 2)} MB")
print(f" - Checkpoint Hash: {ckpt_hash}")

assert ckpt_hash == EXPECTED_CKPT_HASH, f"Checkpoint hash mismatch! Expected {EXPECTED_CKPT_HASH}, got {ckpt_hash}"
print("✅ SHA-256 matches the verified Epoch 5 baseline checkpoint!")

# Verify all 30 class names match dataset config exactly
expected_names = dataset_cfg['names']
for cid, cname in expected_names.items():
    actual_name = model.names.get(cid)
    assert actual_name == cname, f"Class ID {cid} mismatch: '{actual_name}' != '{cname}'"

print("✅ All 30 class IDs and names match dataset specification perfectly!")
print("✅ Checkpoint is fully compatible with Ultralytics YOLOv8 GPU training.")


---
## 5. Proposed GPU Fine-Tuning Execution
This cell executes the fine-tuning run.

> **SAFETY GUARD**:
> Set `START_TRAINING = True` to confirm user authorization before launching.
> - **Epochs**: 20 additional epochs on GPU.
> - **Resolution**: `imgsz=640` (restores spatial resolution for grapes, berries, candies).
> - **Batch Size**: `batch=16` (conservative for T4 15–16 GB VRAM).
> - **Estimated Duration**: ~15–20 minutes on Colab T4.
> - **Cost**: $0.00 (Free Tier).


In [ ]:
# Cell 5: Fine-Tuning Execution (Requires Explicit User Confirmation)
import datetime
import shutil
import hashlib
import os
from pathlib import Path
from ultralytics import YOLO

# =====================================================================
# SAFETY GUARD: Training is disabled by default to prevent accidental runs.
# Change START_TRAINING = True only when you are ready to train.
# =====================================================================
START_TRAINING = False

def compute_file_sha256(filepath):
    """Compute SHA-256 hash of a file."""
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192 * 1024):
            h.update(chunk)
    return h.hexdigest().upper()

def atomic_copy_to_drive(src_file: Path, dest_file: Path):
    """
    Safely copies a file to Google Drive using a temporary .tmp file
    and atomic replacement to prevent exposing partially written checkpoints
    or corrupting existing valid backups on sudden runtime disconnects.
    """
    if not src_file.exists():
        return False
    
    dest_file.parent.mkdir(parents=True, exist_ok=True)
    tmp_dest = dest_file.with_name(f"{dest_file.name}.tmp")
    
    # 1. Copy to temporary file
    shutil.copy2(src_file, tmp_dest)
    
    # 2. Verify copied file size
    src_size = src_file.stat().st_size
    tmp_size = tmp_dest.stat().st_size
    if src_size != tmp_size:
        if tmp_dest.exists():
            tmp_dest.unlink()
        raise IOError(f"Sync verification failed for {src_file.name}: size mismatch ({src_size} vs {tmp_size} bytes)")
    
    # 3. Atomic replacement to final target
    os.replace(str(tmp_dest), str(dest_file))
    return True

if not START_TRAINING:
    print("=" * 65)
    print("⏸️  TRAINING EXECUTION PAUSED BY DEFAULT")
    print("To launch GPU fine-tuning, change START_TRAINING = True in this cell.")
    print("=" * 65)
else:
    # POLICY ASSERTIONS:
    # 1. Strictly free T4 GPU budget ($0.00)
    # 2. Zero modifications to production best.pt
    # 3. Held-out 300 test images remain completely untouched during training
    print("=" * 65)
    print("🔒 VERIFYING PRE-TRAINING SAFETY & ISOLATION GUARDS:")
    print(" - Target Device: Free GPU (cuda:0)")
    print(" - Production Path: Isolated (writing exclusively to Google Drive experiments)")
    print(" - Held-Out Test Set: Isolated (300 test images excluded from model.train)")
    print("=" * 65)

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    EXPERIMENT_NAME = f"yolov8n_food_gpu_finetune_20ep_{timestamp}"
    
    # Save local experiment run on /content
    LOCAL_RUNS_DIR = Path('/content/runs')
    EXPERIMENT_LOCAL_DIR = LOCAL_RUNS_DIR / EXPERIMENT_NAME
    
    # Unique timestamped Drive destination (guarantees no overwriting of unrelated experiments)
    EXPERIMENT_DRIVE_DIR = DRIVE_EXPERIMENTS_DIR / EXPERIMENT_NAME
    (EXPERIMENT_DRIVE_DIR / 'weights').mkdir(parents=True, exist_ok=True)
    
    print(f"🚀 Launching GPU Fine-Tuning: {EXPERIMENT_NAME}")
    print(f"📁 Local NVMe Dir: {EXPERIMENT_LOCAL_DIR}")
    print(f"📁 Drive Sync Dir: {EXPERIMENT_DRIVE_DIR}")
    print(f"⚙️  Batch Size:     {RECOMMENDED_BATCH_SIZE}")
    print(f"⚙️  Image Size:     {RECOMMENDED_IMGSZ} px")
    print("=" * 65)
    
    # Load model from verified Epoch 5 checkpoint
    model = YOLO(str(CKPT_SOURCE))
    
    # Progressive checkpoint sync tracking
    sync_tracker = {
        'last_synced_epoch': None,
        'sync_errors': []
    }
    
    # Ultralytics lifecycle: 'on_fit_epoch_end' runs after save_metrics() (results.csv)
    # and save_model() (weights/last.pt & weights/best.pt) are completely written.
    def on_train_epoch_end_sync(trainer):
        epoch_idx = getattr(trainer, 'epoch', -1) + 1
        try:
            local_w = Path(trainer.save_dir) / 'weights'
            drive_w = EXPERIMENT_DRIVE_DIR / 'weights'
            
            synced_items = []
            for fname in ['last.pt', 'best.pt']:
                src = local_w / fname
                if src.exists():
                    atomic_copy_to_drive(src, drive_w / fname)
                    synced_items.append(fname)
            
            results_csv = Path(trainer.save_dir) / 'results.csv'
            if results_csv.exists():
                atomic_copy_to_drive(results_csv, EXPERIMENT_DRIVE_DIR / 'results.csv')
                synced_items.append('results.csv')
                
            sync_tracker['last_synced_epoch'] = epoch_idx
            print(f" 💾 [Drive Sync Ep {epoch_idx:2d}/20] Verified & atomic-synced: {', '.join(synced_items)}")
        except Exception as e:
            err_msg = f"Drive sync error at epoch {epoch_idx}: {e}"
            sync_tracker['sync_errors'].append(err_msg)
            print(f" ⚠️ {err_msg}")
    
    # Register the verified on_fit_epoch_end callback
    model.add_callback("on_fit_epoch_end", on_train_epoch_end_sync)
    
    # Execute fine-tuning (train and val splits only; test split is strictly excluded)
    results = model.train(
        data=str(colab_yaml_path),
        epochs=20,
        imgsz=RECOMMENDED_IMGSZ,
        batch=RECOMMENDED_BATCH_SIZE,
        device=0,           # Free GPU 0
        workers=2,          # Optimal for Colab 2-core CPU loader
        project=str(LOCAL_RUNS_DIR),
        name=EXPERIMENT_NAME,
        seed=42,
        deterministic=True,
        close_mosaic=5,     # Natural non-mosaic fine-tuning for final 5 epochs
        save=True,
        save_period=5,      # Save checkpoint every 5 epochs
        val=True,           # Evaluate strictly on validation split at each epoch
        plots=True,
        verbose=True
    )
    
    print("\n" + "=" * 65)
    print("🎉 TRAINING COMPLETED SUCCESSFULLY!")
    print("=" * 65)
    
    # Safely copy all remaining training artifacts (curves, plots, logs) to Google Drive
    print(f"Preserving all experiment artifacts to Drive: {EXPERIMENT_DRIVE_DIR}...")
    copied_count = 0
    for local_file in EXPERIMENT_LOCAL_DIR.rglob('*'):
        if local_file.is_file():
            rel_path = local_file.relative_to(EXPERIMENT_LOCAL_DIR)
            dest_file = EXPERIMENT_DRIVE_DIR / rel_path
            atomic_copy_to_drive(local_file, dest_file)
            copied_count += 1
            
    print(f"✅ Successfully preserved {copied_count} artifact files to Drive.")
    
    # Final checkpoint integrity and SHA-256 hash match verification
    local_best = EXPERIMENT_LOCAL_DIR / 'weights' / 'best.pt'
    drive_best = EXPERIMENT_DRIVE_DIR / 'weights' / 'best.pt'
    
    if not drive_best.exists():
        raise FileNotFoundError(f"CRITICAL: Failed to locate backed up best.pt at {drive_best}!")
        
    local_hash = compute_file_sha256(local_best)
    drive_hash = compute_file_sha256(drive_best)
    
    if local_hash != drive_hash:
        raise ValueError(f"CRITICAL: Checkpoint SHA-256 hash mismatch! Local: {local_hash}, Drive: {drive_hash}")
        
    print(f"✅ Verified Drive Checkpoint: {drive_best}")
    print(f"✅ Verified SHA-256 Match:  {drive_hash}")
    
    if sync_tracker['sync_errors']:
        print(f"⚠️ Note: {len(sync_tracker['sync_errors'])} Drive sync warning(s) occurred during training:")
        for err in sync_tracker['sync_errors']:
            print(f"   • {err}")
    else:
        print("✅ Zero sync errors: All intermediate epoch backups succeeded seamlessly.")


---
## 6. Runtime-Disconnect Recovery & Seamless Resume
If Colab disconnected during training, this cell scans all saved runs in Google Drive and can seamlessly resume where it left off.

> **SAFETY GUARD**:
> Set RESUME_TRAINING = True to resume an interrupted run.
> - **Resumes from**: The exact last.pt checkpoint of the interrupted run.
> - **Will NOT restart**: It will NOT restart from the Epoch 5 baseline, preventing any loss of completed epochs.
> - **Drive Persistence**: Automatically registers the atomic Drive sync callback for all remaining epochs.


In [ ]:
# Cell 6: Disconnect Recovery & Experiment Inspector (Guarded)
from pathlib import Path
from ultralytics import YOLO
import os

# =====================================================================
# SAFETY GUARD: Disconnect Recovery / Resume Training
# Change RESUME_TRAINING = True ONLY if you want to resume an interrupted run.
# Change SELECTED_RECOVERY_RUN = "<folder_name>" to target a specific run,
# or leave as "latest" to auto-target the most recent valid incomplete run.
# =====================================================================
RESUME_TRAINING = False
SELECTED_RECOVERY_RUN = "latest"

print("=" * 65)
print("🔍 Scanning Google Drive for FoodShareAI Experiment Checkpoints...")
print("=" * 65)

runs = sorted([p for p in DRIVE_EXPERIMENTS_DIR.glob('*') if p.is_dir()], key=os.path.getmtime, reverse=True)

classified_runs = {
    'completed': [],
    'incomplete': [],
    'invalid': []
}

for r in runs:
    best_pt = r / 'weights' / 'best.pt'
    last_pt = r / 'weights' / 'last.pt'
    results_csv = r / 'results.csv'
    
    # Check weight files validity (must exist and be > 1 MB)
    has_valid_best = best_pt.exists() and best_pt.stat().st_size > 1024 * 1024
    has_valid_last = last_pt.exists() and last_pt.stat().st_size > 1024 * 1024
    has_csv = results_csv.exists() and results_csv.stat().st_size > 0
    
    epochs_logged = 0
    if has_csv:
        try:
            with open(results_csv, 'r', encoding='utf-8') as f:
                lines = [l for l in f.readlines() if l.strip()]
            epochs_logged = max(0, len(lines) - 1)
        except Exception:
            epochs_logged = 0

    run_info = {
        'dir': r,
        'name': r.name,
        'best_pt': best_pt if has_valid_best else None,
        'last_pt': last_pt if has_valid_last else None,
        'results_csv': results_csv if has_csv else None,
        'epochs_logged': epochs_logged,
        'best_size_mb': round(best_pt.stat().st_size / (1024*1024), 2) if has_valid_best else 0,
        'last_size_mb': round(last_pt.stat().st_size / (1024*1024), 2) if has_valid_last else 0,
    }
    
    # Classification rules:
    if epochs_logged >= 20 and has_valid_best and has_valid_last:
        classified_runs['completed'].append(run_info)
    elif epochs_logged < 20 and has_valid_last and has_csv:
        classified_runs['incomplete'].append(run_info)
    else:
        classified_runs['invalid'].append(run_info)

# Display inventory report
print(f"Total Experiment Directories Found: {len(runs)}\n")

if classified_runs['completed']:
    print("🟢 COMPLETED RUNS (20/20 Epochs):")
    for r in classified_runs['completed']:
        print(f"  • {r['name']} — best.pt ({r['best_size_mb']} MB), last.pt ({r['last_size_mb']} MB)")
    print()

if classified_runs['incomplete']:
    print("🟡 INCOMPLETE RUNS (Recoverable):")
    for r in classified_runs['incomplete']:
        print(f"  • {r['name']} — Epochs: {r['epochs_logged']}/20, last.pt ({r['last_size_mb']} MB)")
    print()

if classified_runs['invalid']:
    print("🔴 INVALID / CORRUPTED DIRECTORIES:")
    for r in classified_runs['invalid']:
        reason = "Missing/empty last.pt" if not r['last_pt'] else "Missing/empty results.csv"
        print(f"  • {r['name']} — Issue: {reason}")
    print()

# Select recovery target
recovery_target = None
if classified_runs['incomplete']:
    if SELECTED_RECOVERY_RUN == "latest":
        recovery_target = classified_runs['incomplete'][0]
    else:
        for r in classified_runs['incomplete']:
            if r['name'] == SELECTED_RECOVERY_RUN:
                recovery_target = r
                break

if not RESUME_TRAINING:
    print("=" * 65)
    print("⏸️  DISCONNECT RECOVERY EXECUTION PAUSED BY DEFAULT")
    if recovery_target:
        print(f"💡 Selected Recovery Target: {recovery_target['name']}")
        print(f"   Last saved epoch: {recovery_target['epochs_logged']}/20")
        print(f"   Checkpoint path:  {recovery_target['last_pt']}")
        print("\nTo resume this exact run from where it stopped:")
        print(" 1. Verify SELECTED_RECOVERY_RUN is set to the intended run.")
        print(" 2. Change RESUME_TRAINING = True in this cell and run it.")
        print(" (Resuming will continue strictly from last.pt; it will NEVER restart from Epoch 5).")
    elif classified_runs['completed'] and not classified_runs['incomplete']:
        print("🎉 All existing runs are fully completed (20/20 epochs). No recovery needed!")
    else:
        print("No recoverable incomplete runs found in Google Drive.")
    print("=" * 65)
else:
    if not recovery_target:
        raise FileNotFoundError(
            f"RESUME_TRAINING is True, but recovery target '{SELECTED_RECOVERY_RUN}' "
            "is not a valid incomplete run with a verified last.pt!"
        )
    
    ckpt_path = recovery_target['last_pt']
    print("=" * 65)
    print(f"🔄 RESUMING INTERRUPTED RUN: {recovery_target['name']}")
    print(f"📁 Checkpoint Source: {ckpt_path}")
    print(f"📍 Resuming from epoch {recovery_target['epochs_logged']} to 20 on Free GPU...")
    print("=" * 65)
    
    # Load strictly from the interrupted run's last.pt (never baseline)
    resume_model = YOLO(str(ckpt_path))
    
    # Attach atomic sync callback for remaining epochs
    EXPERIMENT_DRIVE_DIR = recovery_target['dir']
    
    def on_resume_epoch_end_sync(trainer):
        try:
            local_w = Path(trainer.save_dir) / 'weights'
            drive_w = EXPERIMENT_DRIVE_DIR / 'weights'
            for fname in ['last.pt', 'best.pt']:
                src = local_w / fname
                if src.exists():
                    atomic_copy_to_drive(src, drive_w / fname)
            results_csv = Path(trainer.save_dir) / 'results.csv'
            if results_csv.exists():
                atomic_copy_to_drive(results_csv, EXPERIMENT_DRIVE_DIR / 'results.csv')
        except Exception as e:
            print(f"⚠️ Resume Drive sync error: {e}")
            
    resume_model.add_callback("on_fit_epoch_end", on_resume_epoch_end_sync)
    
    # Resume training directly
    resume_results = resume_model.train(resume=True)
    print("\n🎉 Resumed training run completed successfully!")


---
## 7. Full Validation Evaluation & Baseline Comparison
Evaluates the candidate checkpoint against the full 589-image validation split and compares with the Epoch 5 baseline.


In [ ]:
# Cell 7: Full Validation Set Evaluation
from ultralytics import YOLO
import json
import hashlib
from pathlib import Path
import datetime
import os

# =====================================================================
# SAFETY GUARD: Validation evaluation is disabled by default.
# Set RUN_VALIDATION_EVAL = True to evaluate on the 589 validation images.
# Set CANDIDATE_CHECKPOINT_PATH = "latest" to evaluate latest trained run,
# or specify the exact Path to best.pt in Drive.
# =====================================================================
RUN_VALIDATION_EVAL = False
CANDIDATE_CHECKPOINT_PATH = "latest"

def compute_file_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192 * 1024):
            h.update(chunk)
    return h.hexdigest().upper()

# Baseline Epoch 5 metrics (verified from full_validation_report.json)
BASELINE_METRICS = {
    'checkpoint_source': 'Epoch 5 CPU Baseline (yolov8n_10ep_416_cpu)',
    'sha256': '1DA8CD2C9CC6911108DA61456B5E5FC7D6BD84D1C79A24DC18E2321A602FAF6B',
    'precision': 0.2430,
    'recall': 0.2437,
    'mAP50': 0.1991,
    'mAP50_95': 0.1562
}

if not RUN_VALIDATION_EVAL:
    print("=" * 65)
    print("⏸️  VALIDATION EVALUATION PAUSED BY DEFAULT")
    print("To evaluate a trained candidate model on the 589 validation images:")
    print(" 1. Ensure fine-tuning (Cell 5 or Cell 6) has completed.")
    print(" 2. Set CANDIDATE_CHECKPOINT_PATH = 'latest' or specify the exact path.")
    print(" 3. Change RUN_VALIDATION_EVAL = True in this cell and run it.")
    print("=" * 65)
else:
    # Resolve explicit candidate checkpoint
    target_ckpt = None
    if CANDIDATE_CHECKPOINT_PATH == "latest":
        runs = sorted([p for p in DRIVE_EXPERIMENTS_DIR.glob('*') if p.is_dir()], key=os.path.getmtime, reverse=True)
        for r in runs:
            candidate_file = r / 'weights' / 'best.pt'
            if candidate_file.exists() and candidate_file.stat().st_size > 1024 * 1024:
                target_ckpt = candidate_file
                break
    else:
        p = Path(CANDIDATE_CHECKPOINT_PATH)
        if p.exists() and p.stat().st_size > 1024 * 1024:
            target_ckpt = p

    if target_ckpt is None:
        raise FileNotFoundError(
            "CRITICAL: No trained candidate checkpoint found! "
            "Cell 7 will NOT silently substitute the Epoch 5 baseline as a candidate. "
            "Please train a model first (Cell 5) or specify a valid candidate path."
        )

    ckpt_hash = compute_file_sha256(target_ckpt)
    print("=" * 65)
    print(f"📊 EVALUATING CANDIDATE CHECKPOINT")
    print(f"📁 Checkpoint Path: {target_ckpt}")
    print(f"🔑 SHA-256 Checksum: {ckpt_hash}")
    print("=" * 65)

    eval_model = YOLO(str(target_ckpt))
    val_results = eval_model.val(
        data=str(colab_yaml_path),
        split='val',
        imgsz=640,
        batch=16,
        device=0
    )

    # Extract metrics
    cand_precision = float(val_results.results_dict.get('metrics/precision(B)', 0))
    cand_recall = float(val_results.results_dict.get('metrics/recall(B)', 0))
    cand_map50 = float(val_results.results_dict.get('metrics/mAP50(B)', 0))
    cand_map50_95 = float(val_results.results_dict.get('metrics/mAP50-95(B)', 0))

    # Persist standalone validation evaluation report
    eval_report = {
        'report_type': 'CANDIDATE_VALIDATION_EVALUATION',
        'timestamp': datetime.datetime.now().isoformat(),
        'checkpoint_evaluated': {
            'path': str(target_ckpt),
            'sha256': ckpt_hash,
            'size_bytes': target_ckpt.stat().st_size
        },
        'dataset_evaluation': {
            'split': 'val',
            'images_evaluated': 589,
            'imgsz': 640
        },
        'candidate_metrics': {
            'precision': round(cand_precision, 4),
            'recall': round(cand_recall, 4),
            'mAP50': round(cand_map50, 4),
            'mAP50_95': round(cand_map50_95, 4)
        },
        'baseline_comparison': {
            'baseline_source': BASELINE_METRICS['checkpoint_source'],
            'baseline_mAP50': BASELINE_METRICS['mAP50'],
            'baseline_mAP50_95': BASELINE_METRICS['mAP50_95'],
            'delta_mAP50': round(cand_map50 - BASELINE_METRICS['mAP50'], 4),
            'delta_mAP50_95': round(cand_map50_95 - BASELINE_METRICS['mAP50_95'], 4)
        }
    }

    eval_report_dir = target_ckpt.parent.parent
    eval_report_file = eval_report_dir / f"validation_eval_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(eval_report_file, 'w', encoding='utf-8') as f:
        json.dump(eval_report, f, indent=2)

    print("\n" + "=" * 65)
    print("📈 VALIDATION COMPARISON RESULTS (589 Images):")
    print("=" * 65)
    print(f" - Precision:   Candidate {cand_precision:.4f}  |  Baseline {BASELINE_METRICS['precision']:.4f}")
    print(f" - Recall:      Candidate {cand_recall:.4f}  |  Baseline {BASELINE_METRICS['recall']:.4f}")
    print(f" - mAP@50:      Candidate {cand_map50:.4f}  |  Baseline {BASELINE_METRICS['mAP50']:.4f}")
    print(f" - mAP@50-95:   Candidate {cand_map50_95:.4f}  |  Baseline {BASELINE_METRICS['mAP50_95']:.4f}")
    print(f"\n💾 Validation report saved to Drive: {eval_report_file}")
    print("=" * 65)


---
## 8. Final Held-Out Test Split Evaluation
> ⚠️ **STRICT EVALUATION PROTOCOL**:
> The 300 test images have **never** been seen by any training loop or hyperparameter search.
> Run this cell **only once** on your chosen final candidate model to confirm real-world generalization before considering production promotion.


In [ ]:
# Cell 8: Final Held-Out Test Split Evaluation (Run Once on Final Candidate)
from ultralytics import YOLO
import json
import hashlib
from pathlib import Path
import datetime

# =====================================================================
# SAFETY GUARD: Test split is strictly held out.
# Set RUN_TEST_EVALUATION = True ONLY for the final selected candidate!
# You MUST provide the explicit path to the winning candidate checkpoint.
# =====================================================================
RUN_TEST_EVALUATION = False
FINAL_CANDIDATE_PATH = None  # e.g., DRIVE_EXPERIMENTS_DIR / "yolov8n_food_gpu_finetune_..." / "weights" / "best.pt"

def compute_file_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192 * 1024):
            h.update(chunk)
    return h.hexdigest().upper()

if not RUN_TEST_EVALUATION:
    print("=" * 65)
    print("⏸️  HELD-OUT TEST EVALUATION PAUSED BY DEFAULT")
    print("Protocol Notice:")
    print(" - The 300 test images are reserved strictly for the single final winner.")
    print(" - Do NOT run test evaluation repeatedly or use it for model selection.")
    print(" - To evaluate your chosen winner:")
    print("   1. Set FINAL_CANDIDATE_PATH = Path('...') pointing to your best.pt.")
    print("   2. Set RUN_TEST_EVALUATION = True in this cell and run it once.")
    print("=" * 65)
else:
    if FINAL_CANDIDATE_PATH is None:
        raise ValueError(
            "CRITICAL PROTOCOL VIOLATION: FINAL_CANDIDATE_PATH cannot be None! "
            "You must explicitly declare the winning candidate checkpoint path to evaluate on the test split."
        )

    final_ckpt = Path(FINAL_CANDIDATE_PATH)
    if not final_ckpt.exists() or final_ckpt.stat().st_size <= 1024 * 1024:
        raise FileNotFoundError(f"Candidate checkpoint not found or invalid at {final_ckpt}!")

    ckpt_hash = compute_file_sha256(final_ckpt)
    print("=" * 65)
    print("🎯 RUNNING FINAL HELD-OUT TEST EVALUATION (300 IMAGES)")
    print(f"📁 Winning Checkpoint: {final_ckpt}")
    print(f"🔑 SHA-256 Checksum:  {ckpt_hash}")
    print("=" * 65)

    test_model = YOLO(str(final_ckpt))
    test_results = test_model.val(
        data=str(colab_yaml_path),
        split='test',
        imgsz=640,
        batch=16,
        device=0
    )

    test_p = float(test_results.results_dict.get('metrics/precision(B)', 0))
    test_r = float(test_results.results_dict.get('metrics/recall(B)', 0))
    test_map50 = float(test_results.results_dict.get('metrics/mAP50(B)', 0))
    test_map50_95 = float(test_results.results_dict.get('metrics/mAP50-95(B)', 0))

    # Persist standalone test evaluation report
    test_report = {
        'report_type': 'FINAL_HELD_OUT_TEST_EVALUATION',
        'timestamp': datetime.datetime.now().isoformat(),
        'checkpoint_evaluated': {
            'path': str(final_ckpt),
            'sha256': ckpt_hash,
            'size_bytes': final_ckpt.stat().st_size
        },
        'dataset_evaluation': {
            'split': 'test',
            'images_evaluated': 300,
            'imgsz': 640
        },
        'test_metrics': {
            'precision': round(test_p, 4),
            'recall': round(test_r, 4),
            'mAP50': round(test_map50, 4),
            'mAP50_95': round(test_map50_95, 4)
        }
    }

    test_report_file = final_ckpt.parent.parent / f"held_out_test_eval_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(test_report_file, 'w', encoding='utf-8') as f:
        json.dump(test_report, f, indent=2)

    print("\n" + "=" * 65)
    print("🎯 FINAL HELD-OUT TEST RESULTS (300 Images):")
    print("=" * 65)
    print(f" - Precision:   {test_p:.4f}")
    print(f" - Recall:      {test_r:.4f}")
    print(f" - mAP@50:      {test_map50:.4f}")
    print(f" - mAP@50-95:   {test_map50_95:.4f}")
    print(f"\n💾 Test report saved to Drive: {test_report_file}")
    print("=" * 65)


---
## 9. Final Experiment Report Generation
Generates a structured JSON summary report and saves it directly to Google Drive.


In [ ]:
# Cell 9: Generate Structured Experiment Report
import json
import hashlib
import datetime
from pathlib import Path
import os
import torch

def get_file_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192 * 1024):
            h.update(chunk)
    return h.hexdigest().upper()

# Baseline Epoch 5 metrics verified against ml-service/runs/.../full_validation_report.json
VERIFIED_BASELINE_METRICS = {
    "source_checkpoint": "Epoch 5 CPU Baseline (best.pt)",
    "checkpoint_sha256": "1DA8CD2C9CC6911108DA61456B5E5FC7D6BD84D1C79A24DC18E2321A602FAF6B",
    "evaluation_split": "val (589 images, imgsz 416, CPU)",
    "precision": 0.2430,
    "recall": 0.2437,
    "mAP50": 0.1991,
    "mAP50_95": 0.1562,
    "historical_epoch5_csv_mAP50": 0.1994
}

# Scan Drive for actual persisted experiment runs and evaluation reports
runs = sorted([p for p in DRIVE_EXPERIMENTS_DIR.glob('*') if p.is_dir()], key=os.path.getmtime, reverse=True) if 'DRIVE_EXPERIMENTS_DIR' in globals() else []

latest_run = None
candidate_weights = None
persisted_val_metrics = None
persisted_test_metrics = None

if runs:
    latest_run = runs[0]
    best_candidate = latest_run / 'weights' / 'best.pt'
    if best_candidate.exists() and best_candidate.stat().st_size > 1024 * 1024:
        candidate_weights = best_candidate
        
        # Look for persisted validation evaluation JSON in candidate folder
        val_reports = sorted(latest_run.glob('validation_eval_*.json'), key=os.path.getmtime, reverse=True)
        if val_reports:
            try:
                with open(val_reports[0], 'r', encoding='utf-8') as f:
                    vdata = json.load(f)
                    persisted_val_metrics = vdata.get('candidate_metrics')
            except Exception:
                pass
                
        # Look for persisted test evaluation JSON in candidate folder
        test_reports = sorted(latest_run.glob('held_out_test_eval_*.json'), key=os.path.getmtime, reverse=True)
        if test_reports:
            try:
                with open(test_reports[0], 'r', encoding='utf-8') as f:
                    tdata = json.load(f)
                    persisted_test_metrics = tdata.get('test_metrics')
            except Exception:
                pass

# Determine truthful report status
training_has_occurred = candidate_weights is not None
validation_has_occurred = persisted_val_metrics is not None
test_has_occurred = persisted_test_metrics is not None

if not training_has_occurred:
    report_status = "PRE_TRAINING_PREFLIGHT_ONLY"
    status_summary = "Preflight completed. Training has NOT been executed yet. Candidate metrics are null."
elif not validation_has_occurred:
    report_status = "TRAINING_COMPLETED_AWAITING_VALIDATION"
    status_summary = "Fine-tuning completed. Validation evaluation (Cell 7) has NOT been run yet."
else:
    report_status = "EXPERIMENT_EVALUATED"
    status_summary = "Fine-tuning and validation evaluation completed and verified from persisted artifacts."

report_data = {
    "project": "FoodShareAI",
    "report_type": report_status,
    "summary": status_summary,
    "timestamp": datetime.datetime.now().isoformat(),
    "execution_state": {
        "training_executed": training_has_occurred,
        "validation_evaluated": validation_has_occurred,
        "held_out_test_evaluated": test_has_occurred
    },
    "hardware_environment": {
        "gpu_allocated": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
        "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if torch.cuda.is_available() else 0,
        "budget_policy": "Strictly Free Tier ($0.00)"
    },
    "verified_baseline_epoch5": VERIFIED_BASELINE_METRICS,
    "candidate_checkpoint": {
        "run_folder": latest_run.name if latest_run else None,
        "checkpoint_path": str(candidate_weights) if candidate_weights else None,
        "checkpoint_sha256": get_file_sha256(candidate_weights) if candidate_weights else None,
        "validation_metrics": persisted_val_metrics,
        "held_out_test_metrics": persisted_test_metrics
    },
    "production_safety_check": {
        "production_best_pt_modified": False,
        "production_slot_preserved": True,
        "model_promoted": False
    }
}

timestamp_str = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
report_out_path = DRIVE_PROJECT_ROOT / f"foodshareai_report_{report_status.lower()}_{timestamp_str}.json"
with open(report_out_path, 'w', encoding='utf-8') as f:
    json.dump(report_data, f, indent=2)

print("=" * 65)
print(f"📄 FOODSHAREAI STRUCTURED REPORT ({report_status})")
print("=" * 65)
print(f"📁 Report saved to Drive: {report_out_path}")
print(f"ℹ️  Status: {status_summary}\n")
print(json.dumps(report_data, indent=2))
print("=" * 65)
